# 03 — Gold Aggregation
Creates the curated tables used by forecasting and the grid-operations agent.

**Input:** `pgcb.silver.grid_clean`  
**Outputs:**
- `pgcb.gold.daily_demand`
- `pgcb.gold.grid_operations`

In [ ]:
from pyspark.sql import functions as F

SILVER_TABLE = "pgcb.silver.grid_clean"
DAILY_DEMAND_TABLE = "pgcb.gold.daily_demand"
GRID_OPERATIONS_TABLE = "pgcb.gold.grid_operations"

silver_df = spark.table(SILVER_TABLE)
spark.sql("CREATE SCHEMA IF NOT EXISTS pgcb.gold")

## Daily demand table

In [ ]:
daily_demand_df = (
    silver_df
    .groupBy("date")
    .agg(
        F.avg("demand_mw").alias("avg_demand_mw"),
        F.max("demand_mw").alias("peak_demand_mw"),
        F.min("demand_mw").alias("min_demand_mw"),
        F.avg("generation_mw").alias("avg_generation_mw"),
        F.avg("load_shedding").alias("avg_load_shedding_mw"),
        F.count("*").alias("observation_count")
    )
    .orderBy("date")
)

(
    daily_demand_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(DAILY_DEMAND_TABLE)
)

## Grid operations table

In [ ]:
grid_operations_df = silver_df.select(
    "datetime", "date", "generation_mw", "demand_mw", "load_shedding",
    "domestic_generation_mw", "total_imports_mw", "calculated_supply_mw",
    "supply_demand_gap_mw"
)

(
    grid_operations_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GRID_OPERATIONS_TABLE)
)

## Gold validation

In [ ]:
spark.sql(f"""
SELECT
    MIN(date) AS start_date,
    MAX(date) AS end_date,
    COUNT(*) AS number_of_days,
    SUM(CASE WHEN avg_demand_mw IS NULL THEN 1 ELSE 0 END) AS missing_daily_demand
FROM {DAILY_DEMAND_TABLE}
""").show()

display(
    spark.sql(f"""
        SELECT *
        FROM {DAILY_DEMAND_TABLE}
        ORDER BY date DESC
        LIMIT 20
    """)
)